# DIPAI quickstart notebook

A guided tour of the repository **without downloading anything or training
anything long**: we inspect the licensed data registry, look at what a
prepared dataset contains, evaluate a small existing checkpoint on a couple
of examples, and list registered models.

Every cell runs in-process against the local repo (no shell magics), so the
notebook is verifiable with `python notebooks/verify_notebook.py`.

Commands that *would* touch the network or the GPU are shown as markdown
blocks for you to run in a terminal when you mean them.

In [ ]:
import sys
from pathlib import Path

# repo root on sys.path, whether launched from repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "dipai").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import torch, transformers
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# Hardware probe: what this machine has to work with (read-only, no model load).
from dipai.hardware import format_report, probe_hardware

hw = probe_hardware()
print(format_report(hw))

## Data pipeline (run in a terminal, not here)

Bounded, licensed, resumable collection → deterministic cleaning → tokenize
+ pack. The caps in `configs/data.yaml` are deliberately tiny pilots; nothing
downloads a multi-terabyte corpus.

```bat
python -m dipai.data.collect --config configs\data.yaml --list
python -m dipai.data.collect --config configs\data.yaml --dry-run
python -m dipai.data.collect --config configs\data.yaml

python -m dipai.data.clean --input data\raw\web\*.jsonl --output data\clean\clean.jsonl --report data\clean\report.json --markdown

python -m dipai.data.prepare --inputs data\clean\clean.jsonl --output-dir data\processed ^
    --tokenizer data\processed\tokenizer.model --block-size 256 --seed 1337
```

Licences for every source: `docs/DATA_SOURCES_AND_LICENSES.md`.

In [ ]:
# The data registry is pure metadata - listing it never touches the network.
from dipai.data.registry import default_registry

reg = default_registry()
print(f"{'name':24s} {'kind':16s} {'licence':26s} pilot")
for name in reg.names():
    s = reg.get(name)
    print(f"{name:24s} {s.kind:16s} {s.license.spdx:26s} {'yes' if s.pilot_default else '-'}")
print()
print("pilot sources:", [s.name for s in reg.pilot()])

## What a prepared dataset looks like

`dipai.data.prepare` produces `train.bin` / `val.bin` (packed uint16 tokens),
`dataset_stats.json`, a tokenizer, and a `manifest.json` recording exactly
what went in — inputs, seed, block size, counts, cleaning report. Training
runs are reproducible from that manifest.

To (re)build one locally, run the `dipai.data.prepare` command above and the
smaller helper scripts (`scripts/prepare_data.py`).

In [ ]:
# Inspect what is already prepared in this repo (guarded: present-or-skipped).
import json

stats_p = ROOT / "data" / "processed" / "dataset_stats.json"
manifest_p = ROOT / "data" / "processed" / "manifest.json"

if stats_p.exists():
    stats = json.loads(stats_p.read_text(encoding="utf-8"))
    print("dataset_stats.json:")
    for k, v in stats.items():
        print(f"  {k}: {v}")
else:
    print("no data/processed/dataset_stats.json yet - run the prepare step")

print()
if manifest_p.exists():
    manifest = json.loads(manifest_p.read_text(encoding="utf-8"))
    print("manifest.json keys:", sorted(manifest)[:12])
else:
    print("no manifest.json yet (older datasets predate the prepare pipeline)")

## Evaluate a small checkpoint

Perplexity = exp(mean cross-entropy) over held-out text. Comparable only
across the **same eval file and tokenizer** — `dipai.evaluate` records and
warns about mismatches. Here we score just 2 documents of the existing
`finetune.jsonl` against the tiny `checkpoints/smoke` model so the notebook
stays fast (seconds on CPU).

The real workflow (frozen eval set + baseline comparison) lives in
`docs/EVALUATION.md`.

In [ ]:
# Fast eval: 2 examples, short context, existing tiny checkpoint.
import tempfile
from pathlib import Path

from dipai.evaluate import evaluate_checkpoint, write_report

smoke = ROOT / "checkpoints" / "smoke"
eval_jsonl = ROOT / "data" / "processed" / "finetune.jsonl"

if not (smoke.exists() and eval_jsonl.exists()):
    print("skipped: needs checkpoints/smoke and data/processed/finetune.jsonl")
else:
    result = evaluate_checkpoint(
        smoke, eval_jsonl, device="cpu",
        max_length=64, batch_size=2, limit=2,
    )
    with tempfile.TemporaryDirectory() as td:
        out = Path(td) / "eval.json"
        write_report(result, out)
        print("report written:", len(out.read_text()), "bytes")
    print(f"backend={result.backend} examples={result.num_examples} "
          f"tokens={result.num_tokens} loss={result.mean_loss:.4f} "
          f"ppl={result.perplexity:.2f}")

## Models you can load

Both CLIs accept registry names or raw paths (`--model` > `--adapter` >
`--ckpt`):

```bat
python -m dipai.chat --list-models
python -m dipai.chat --ckpt checkpoints\scratch --once "your prompt"
python -m webui.app                       :: streaming chat + vision tabs
```

In [ ]:
# The model registry: logical name -> (backend, path). No model is loaded here.
from dipai.model_registry import default_registry

mreg = default_registry()
print(f"{'name':16s} {'backend':8s} {'on disk':8s} path")
for name in mreg.names():
    m = mreg.get(name)
    print(f"{name:16s} {m.backend:8s} {'yes' if m.exists() else 'no':8s} {m.path}")

## Next steps

| Goal | Command / doc |
|---|---|
| Train from scratch (smoke) | `python -m dipai.train --config configs/smoke_4m.yaml` |
| Full training walkthrough | [`docs/TRAINING_GUIDE.md`](../docs/TRAINING_GUIDE.md) |
| Fine-tune (LoRA/QLoRA) | `python -m dipai.finetune --config configs/finetune.yaml --train` |
| Chat with a checkpoint | `python -m dipai.chat --ckpt checkpoints/scratch` |
| Cloud GPU run | [`docs/CLOUD_GPU_TRAINING.md`](../docs/CLOUD_GPU_TRAINING.md) |
| Checkpoints & resume | [`docs/CHECKPOINTS_AND_RESUMING.md`](../docs/CHECKPOINTS_AND_RESUMING.md) |
| Evaluation reference | [`docs/EVALUATION.md`](../docs/EVALUATION.md) |

Safety rules this repo follows: existing checkpoints are never overwritten,
licensing metadata travels with every dataset, and no secrets ever go into
configs or commits.